Conjunto de Datos 4: monthly-car-sales.csv


# 1. Análisis Exploratorio:

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import warnings
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.linear_model import LinearRegression
warnings.filterwarnings('ignore')

In [18]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

df = pd.read_csv('./monthly-car-sales.csv')

# Convertir la columna Month a datetime
# Asumiendo que el formato es como "1960-01" o similar
df['Month'] = pd.to_datetime(df['Month'])
df = df.sort_values('Month')
df.set_index('Month', inplace=True)

# Crear variables adicionales
df['Year'] = df.index.year
df['MonthNum'] = df.index.month
df['Quarter'] = df.index.quarter

data = df['Sales']

# División entrenamiento/prueba
train_size = int(len(data) * 0.8)
train, test = data[:train_size], data[train_size:]

print(f"Datos de entrenamiento: {len(train)}")
print(f"Datos de prueba: {len(test)}")

Datos de entrenamiento: 86
Datos de prueba: 22


In [19]:
from ydata_profiling import ProfileReport

# Generar reporte
profile = ProfileReport(
    df,
    title="Análisis de Nacimientos",
    explorative=True
)

# Guardar reporte
profile.to_file("./profiling_analisis/reporte4.html")

print("✅ Reporte generado")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 135.92it/s]

✅ Reporte generado


In [20]:

# Estadísticas descriptivas
print("=" * 70)
print("ANÁLISIS DE VENTAS MENSUALES DE AUTOS")
print("=" * 70)
print(f"\nPeríodo analizado: {df.index.min().strftime('%B %Y')} - {df.index.max().strftime('%B %Y')}")
print(f"Total de meses: {len(df)}")
print(f"Años completos: {len(df) // 12}")

print("\nESTADÍSTICAS DESCRIPTIVAS:")
print("-" * 40)
print(f"Ventas promedio mensual: {df['Sales'].mean():,.0f} unidades")
print(f"Desviación estándar: {df['Sales'].std():,.0f} unidades")
print(f"Ventas mínimas: {df['Sales'].min():,} ({df['Sales'].idxmin().strftime('%B %Y')})")
print(f"Ventas máximas: {df['Sales'].max():,} ({df['Sales'].idxmax().strftime('%B %Y')})")
print(f"Rango: {df['Sales'].max() - df['Sales'].min():,} unidades")
print(f"Coeficiente de variación: {(df['Sales'].std() / df['Sales'].mean() * 100):.1f}%")

# Calcular tendencia
X = np.arange(len(df)).reshape(-1, 1)
y = df['Sales'].values
lr = LinearRegression()
lr.fit(X, y)
trend_line = lr.predict(X)
monthly_growth = lr.coef_[0]
print(f"\nCrecimiento promedio mensual: {monthly_growth:.1f} unidades/mes")
print(f"Crecimiento promedio anual: {monthly_growth * 12:.0f} unidades/año")


ANÁLISIS DE VENTAS MENSUALES DE AUTOS

Período analizado: January 1960 - December 1968
Total de meses: 108
Años completos: 9

ESTADÍSTICAS DESCRIPTIVAS:
----------------------------------------
Ventas promedio mensual: 14,595 unidades
Desviación estándar: 4,525 unidades
Ventas mínimas: 5,568 (September 1962)
Ventas máximas: 26,099 (May 1968)
Rango: 20,531 unidades
Coeficiente de variación: 31.0%

Crecimiento promedio mensual: 81.2 unidades/mes
Crecimiento promedio anual: 974 unidades/año


In [21]:
# Crear figura con las 3 visualizaciones más relevantes
fig = plt.figure(figsize=(20, 14))

# ========================================
# 1. SERIE TEMPORAL CON ANÁLISIS COMPLETO
# ========================================
ax1 = plt.subplot(3, 1, 1)

# Plot principal con datos originales
ax1.plot(df.index, df['Sales'], color='navy', linewidth=1.5, alpha=0.8, label='Ventas mensuales')

# Agregar línea de tendencia
ax1.plot(df.index, trend_line, 'r--', linewidth=2, alpha=0.8, label=f'Tendencia ({monthly_growth:.0f} unidades/mes)')

# Media móvil de 12 meses (elimina estacionalidad)
df['MA12'] = df['Sales'].rolling(window=12, center=True).mean()
ax1.plot(df.index, df['MA12'], color='green', linewidth=2.5, label='Media móvil 12 meses')

# Resaltar máximos y mínimos históricos
max_idx = df['Sales'].idxmax()
min_idx = df['Sales'].idxmin()
ax1.scatter(max_idx, df.loc[max_idx, 'Sales'], color='red', s=200, zorder=5, 
           label=f'Máximo: {df.loc[max_idx, "Sales"]:,}')
ax1.scatter(min_idx, df.loc[min_idx, 'Sales'], color='darkred', s=200, zorder=5,
           label=f'Mínimo: {df.loc[min_idx, "Sales"]:,}')

# Sombrear períodos de recesión si hay caídas significativas
# Identificar períodos donde las ventas caen más del 20% respecto al año anterior
df['YoY_Change'] = df['Sales'].pct_change(12) * 100
recession_periods = df[df['YoY_Change'] < -20].index

# Formato y etiquetas
ax1.set_title('Serie Temporal de Ventas Mensuales de Autos', fontsize=18, fontweight='bold', pad=20)
ax1.set_xlabel('Año', fontsize=14)
ax1.set_ylabel('Unidades Vendidas', fontsize=14)
ax1.legend(loc='upper left', fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='both', labelsize=12)

# Formatear eje Y con separadores de miles
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# ========================================
# 2. DESCOMPOSICIÓN ESTACIONAL
# ========================================
# Realizar descomposición
decomposition = seasonal_decompose(df['Sales'], model='multiplicative', period=12)

# Subplot para componentes
ax2 = plt.subplot(3, 2, 3)
ax3 = plt.subplot(3, 2, 4)

# Componente estacional por mes
seasonal_avg = pd.DataFrame({
    'Month': range(1, 13),
    'Seasonal': decomposition.seasonal.groupby(decomposition.seasonal.index.month).mean()
})

# Graficar patrón estacional
bars = ax2.bar(seasonal_avg['Month'], seasonal_avg['Seasonal'], color='coral', alpha=0.8)
ax2.axhline(y=1, color='black', linestyle='--', alpha=0.5)
ax2.set_title('Patrón Estacional (Índice Multiplicativo)', fontsize=16, fontweight='bold')
ax2.set_xlabel('Mes', fontsize=12)
ax2.set_ylabel('Índice Estacional', fontsize=12)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
                     'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'])
ax2.grid(True, alpha=0.3, axis='y')

# Añadir valores encima de las barras
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{height:.3f}', ha='center', va='bottom', fontsize=10)

# Box plot por mes para mostrar variabilidad
monthly_data = []
month_labels = []
for month in range(1, 13):
    month_sales = df[df.index.month == month]['Sales'].values
    monthly_data.append(month_sales)
    month_labels.append(['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 
                        'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'][month-1])

bp = ax3.boxplot(monthly_data, labels=month_labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
    patch.set_alpha(0.7)

ax3.set_title('Distribución de Ventas por Mes', fontsize=16, fontweight='bold')
ax3.set_xlabel('Mes', fontsize=12)
ax3.set_ylabel('Unidades Vendidas', fontsize=12)
ax3.grid(True, alpha=0.3, axis='y')
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# ========================================
# 3. ANÁLISIS DE TENDENCIA Y PRONÓSTICO
# ========================================
ax4 = plt.subplot(3, 1, 3)

# Ventas anuales
annual_sales = df.groupby('Year')['Sales'].agg(['sum', 'mean'])
years = annual_sales.index

# Crear gráfico de barras para totales anuales
bars = ax4.bar(years, annual_sales['sum'], alpha=0.6, color='steelblue', label='Total anual')

# Línea para promedio mensual por año
ax4_twin = ax4.twinx()
line = ax4_twin.plot(years, annual_sales['mean'], color='red', marker='o', 
                     linewidth=3, markersize=8, label='Promedio mensual')

# Calcular y mostrar tasa de crecimiento anual
for i in range(1, len(years)):
    growth = ((annual_sales['sum'].iloc[i] - annual_sales['sum'].iloc[i-1]) / 
              annual_sales['sum'].iloc[i-1] * 100)
    ax4.text(years[i], annual_sales['sum'].iloc[i] + 5000, f'{growth:+.1f}%', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# Formato y etiquetas
ax4.set_title('Evolución Anual de Ventas', fontsize=18, fontweight='bold', pad=20)
ax4.set_xlabel('Año', fontsize=14)
ax4.set_ylabel('Ventas Totales Anuales', fontsize=14, color='steelblue')
ax4_twin.set_ylabel('Promedio Mensual', fontsize=14, color='red')
ax4.tick_params(axis='y', labelcolor='steelblue', labelsize=12)
ax4_twin.tick_params(axis='y', labelcolor='red', labelsize=12)
ax4.grid(True, alpha=0.3)

# Formatear ejes Y
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax4_twin.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

# Leyendas combinadas
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4_twin.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)

# Ajustar espaciado
plt.tight_layout()
plt.savefig('./imagenes/4/CARanalisisexploratorio.png', dpi=300, bbox_inches='tight', facecolor='#f8f9fa')
# Guardar figura
plt.show()

# 2. Promedios Móviles:

In [22]:

# Calcular promedios móviles
ma_3 = train.rolling(window=3).mean()
ma_6 = train.rolling(window=6).mean() 
ma_12 = train.rolling(window=12).mean()

# Extender predicciones para el conjunto de prueba
ma_3_extended = data.rolling(window=3).mean()[train_size:]
ma_6_extended = data.rolling(window=6).mean()[train_size:]
ma_12_extended = data.rolling(window=12).mean()[train_size:]

# Calcular RMSE
rmse_ma3 = np.sqrt(mean_squared_error(test, ma_3_extended))
rmse_ma6 = np.sqrt(mean_squared_error(test, ma_6_extended))
rmse_ma12 = np.sqrt(mean_squared_error(test, ma_12_extended))

# Gráfica Promedios Móviles
plt.figure(figsize=(12, 6))
plt.plot(data.index, data.values, label='Original', alpha=0.7, color='gray')
plt.plot(ma_3.index, ma_3.values, label=f'MA_3 (RMSE: {rmse_ma3:.0f})', linewidth=2)
plt.plot(ma_6.index, ma_6.values, label=f'MA_6 (RMSE: {rmse_ma6:.0f})', linewidth=2)
plt.plot(ma_12.index, ma_12.values, label=f'MA_12 (RMSE: {rmse_ma12:.0f})', linewidth=2)
plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
plt.title('Promedios Móviles - Ventas de Autos')
plt.ylabel('Ventas')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('./imagenes/4/CARpomediomovil.png', dpi=300, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()

print("RMSE Promedios Móviles:")
print(f"MA_3: {rmse_ma3:.0f}")
print(f"MA_6: {rmse_ma6:.0f}")
print(f"MA_12: {rmse_ma12:.0f}")

RMSE Promedios Móviles:
MA_3: 2966
MA_6: 3967
MA_12: 3884


# 3. Alisamiento Exponencial:

In [23]:
# Alisamiento Exponencial Simple
model_simple = ExponentialSmoothing(train, trend=None, seasonal=None)
fit_simple = model_simple.fit()
forecast_simple = fit_simple.forecast(len(test))
rmse_simple = np.sqrt(mean_squared_error(test, forecast_simple))

# Alisamiento Exponencial Doble (Holt)
model_double = ExponentialSmoothing(train, trend='add', seasonal=None)
fit_double = model_double.fit()
forecast_double = fit_double.forecast(len(test))
rmse_double = np.sqrt(mean_squared_error(test, forecast_double))

# Gráfica Alisamiento Exponencial
plt.figure(figsize=(12, 6))
plt.plot(train.index, train.values, label='Train', color='blue')
plt.plot(test.index, test.values, label='Test', color='green')
plt.plot(test.index, forecast_simple, label=f'Simple (RMSE: {rmse_simple:.0f})', linewidth=2)
plt.plot(test.index, forecast_double, label=f'Doble/Holt (RMSE: {rmse_double:.0f})', linewidth=2)
plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
plt.title('Alisamiento Exponencial - Ventas de Autos')
plt.ylabel('Ventas')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('./imagenes/4/CARalisamiento.png', dpi=300, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()

print("RMSE Alisamiento Exponencial:")
print(f"Simple: {rmse_simple:.0f}")
print(f"Doble (Holt): {rmse_double:.0f}")

RMSE Alisamiento Exponencial:
Simple: 7331
Doble (Holt): 8501


# 4. HOLT-WINTERS

In [24]:
# Holt-Winters Aditivo
model_hw_add = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12)
fit_hw_add = model_hw_add.fit()
forecast_hw_add = fit_hw_add.forecast(len(test))
rmse_hw_add = np.sqrt(mean_squared_error(test, forecast_hw_add))

# Holt-Winters Multiplicativo
try:
    model_hw_mult = ExponentialSmoothing(train, trend='add', seasonal='mul', seasonal_periods=12)
    fit_hw_mult = model_hw_mult.fit()
    forecast_hw_mult = fit_hw_mult.forecast(len(test))
    rmse_hw_mult = np.sqrt(mean_squared_error(test, forecast_hw_mult))
    mult_success = True
except:
    mult_success = False
    print("Modelo multiplicativo falló")

# Gráfica Holt-Winters
plt.figure(figsize=(12, 6))
plt.plot(train.index, train.values, label='Train', color='blue')
plt.plot(test.index, test.values, label='Test', color='green')
plt.plot(test.index, forecast_hw_add, label=f'HW Aditivo (RMSE: {rmse_hw_add:.0f})', linewidth=2)
if mult_success:
    plt.plot(test.index, forecast_hw_mult, label=f'HW Multiplicativo (RMSE: {rmse_hw_mult:.0f})', linewidth=2)
plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
plt.title('Holt-Winters - Ventas de Autos')
plt.ylabel('Ventas')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('./imagenes/4/CARholt.png', dpi=300, bbox_inches='tight', facecolor='#f8f9fa')

plt.show()

print("RMSE Holt-Winters:")
print(f"Aditivo: {rmse_hw_add:.0f}")
if mult_success:
    print(f"Multiplicativo: {rmse_hw_mult:.0f}")


RMSE Holt-Winters:
Aditivo: 2199
Multiplicativo: 2086


# 5. SARIMA:

In [25]:
# Búsqueda de parámetros SARIMA
print("Buscando parámetros SARIMA...")
best_aic = float('inf')
best_params = None

for p in range(3):
    for d in range(2):
        for q in range(3):
            for P in range(2):
                for D in range(2):
                    for Q in range(2):
                        try:
                            model = SARIMAX(train, order=(p,d,q), seasonal_order=(P,D,Q,12))
                            fit = model.fit(disp=False)
                            if fit.aic < best_aic:
                                best_aic = fit.aic
                                best_params = ((p,d,q), (P,D,Q,12))
                        except:
                            continue

# Ajustar mejor modelo SARIMA
if best_params:
    order, seasonal_order = best_params
    sarima_model = SARIMAX(train, order=order, seasonal_order=seasonal_order)
    sarima_fit = sarima_model.fit(disp=False)
    sarima_forecast = sarima_fit.forecast(len(test))
    rmse_sarima = np.sqrt(mean_squared_error(test, sarima_forecast))
    
    # Gráfica SARIMA
    plt.figure(figsize=(12, 6))
    plt.plot(train.index, train.values, label='Train', color='blue')
    plt.plot(test.index, test.values, label='Test', color='green')
    plt.plot(test.index, sarima_forecast, label=f'SARIMA{order}x{seasonal_order} (RMSE: {rmse_sarima:.0f})', linewidth=2)
    plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
    plt.title('SARIMA - Ventas de Autos')
    plt.ylabel('Ventas')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('./imagenes/4/CARsarima.png', dpi=300, bbox_inches='tight', facecolor='#f8f9fa')
    plt.show()
    
    print(f"Mejores parámetros SARIMA: {best_params}")
    print(f"RMSE SARIMA: {rmse_sarima:.0f}")


Buscando parámetros SARIMA...
Mejores parámetros SARIMA: ((1, 1, 1), (1, 1, 0, 12))
RMSE SARIMA: 1874


# 6. Prophet:

In [26]:
try:
    from prophet import Prophet
    
    # Preparar datos para Prophet
    prophet_data = data.reset_index()
    prophet_data.columns = ['ds', 'y']
    prophet_train = prophet_data[:train_size]
    
    # Crear y entrenar modelo Prophet
    prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    prophet_model.fit(prophet_train)
    
    # Hacer predicciones
    future = prophet_model.make_future_dataframe(periods=len(test), freq='MS')
    prophet_forecast = prophet_model.predict(future)
    prophet_pred = prophet_forecast['yhat'][train_size:].values
    rmse_prophet = np.sqrt(mean_squared_error(test, prophet_pred))
    
    # Gráfica Prophet
    plt.figure(figsize=(12, 6))
    plt.plot(train.index, train.values, label='Train', color='blue')
    plt.plot(test.index, test.values, label='Test', color='green')
    plt.plot(test.index, prophet_pred, label=f'Prophet (RMSE: {rmse_prophet:.0f})', linewidth=2)
    plt.axvline(x=data.index[train_size], color='red', linestyle='--', label='División Train/Test')
    plt.title('Prophet - Ventas de Autos')
    plt.ylabel('Ventas')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('./imagenes/4/CARProphet.png', dpi=300, bbox_inches='tight', facecolor='#f8f9fa')
    plt.show()
    
    print(f"RMSE Prophet: {rmse_prophet:.0f}")
    
except ImportError:
    print("Prophet no está instalado. Instalar con: pip install prophet")


12:32:18 - cmdstanpy - INFO - Chain [1] start processing
12:32:19 - cmdstanpy - INFO - Chain [1] done processing


RMSE Prophet: 1825


# 7. Comparación y Evaluación:


In [27]:
# Recopilar todos los RMSE
all_rmse = {
    'MA_3': rmse_ma3,
    'MA_6': rmse_ma6,
    'MA_12': rmse_ma12,
    'Exp_Simple': rmse_simple,
    'Exp_Doble': rmse_double,
    'HW_Aditivo': rmse_hw_add
}

if mult_success:
    all_rmse['HW_Multiplicativo'] = rmse_hw_mult

if 'rmse_sarima' in locals():
    all_rmse['SARIMA'] = rmse_sarima

if 'rmse_prophet' in locals():
    all_rmse['Prophet'] = rmse_prophet

# Gráfica comparación de errores
plt.figure(figsize=(12, 6))
models = list(all_rmse.keys())
rmse_values = list(all_rmse.values())
bars = plt.bar(models, rmse_values, color='skyblue', edgecolor='black')

# Resaltar el mejor modelo
min_idx = rmse_values.index(min(rmse_values))
bars[min_idx].set_color('green')

plt.title('Comparación de RMSE por Modelo - Ventas de Autos')
plt.ylabel('RMSE')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')

# Añadir valores encima de las barras
for i, v in enumerate(rmse_values):
    plt.text(i, v + max(rmse_values)*0.01, f'{v:.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('./imagenes/4/CARcomparacion.png', dpi=300, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()

print("COMPARACIÓN FINAL DE RMSE:")
for model, rmse in sorted(all_rmse.items(), key=lambda x: x[1]):
    print(f"{model}: {rmse:.0f}")

COMPARACIÓN FINAL DE RMSE:
Prophet: 1825
SARIMA: 1874
HW_Multiplicativo: 2086
HW_Aditivo: 2199
MA_3: 2966
MA_12: 3884
MA_6: 3967
Exp_Simple: 7331
Exp_Doble: 8501
